# O que é o Pré-processamento

É a etapa onde preparamos os dados para que os modelos
de Machine Learning consigam consumir e aprender com eles.

Os modelos trabalham com numeros e não entendem "Yes/No", "Sales Executive" ou "Jovem/Adulto/Senior", na qual são variáveis categóricas. Nessa etapa realizaremos: 
- Encondings
- Separação entre feature e target
- Tratamento do desbalanceamento
- Divisão entre treino/teste

In [3]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Carregar dataset com features
df = pd.read_csv('../data/processed/ibm_hr_analytics_feature_engineering.csv')
print(f"Shape: {df.shape}")
df.head()

Shape: (1000000, 43)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,...,CriticalSatisfactionFlag,RatioCareerCompany,RoleStagnation,YearsWithoutPromotion,AgeGroup,IncomePerLevel,IncomeMedianJob,OverTime_encoded,RiskOvertimeDistance,RiskWorklifeOvertime
0,56,No,Travel_Rarely,590,Research & Development,19,1,Other,4,Female,...,0,0.852941,0.366667,0.366667,Senior,4126.00,1.175722,1,19,3
1,46,No,Travel_Rarely,1441,Human Resources,5,1,Life Sciences,4,Male,...,0,0.973684,0.157895,0.131579,Senior,9987.00,0.951324,1,5,2
2,32,Yes,Travel_Frequently,748,Research & Development,2,3,Medical,4,Male,...,0,0.592593,0.176471,0.882353,Adulto,638.00,0.121478,0,0,0
3,60,No,Travel_Rarely,1311,Human Resources,3,4,Life Sciences,2,Female,...,0,0.966667,0.566667,0.200000,Senior,451.75,0.173001,0,0,0
4,25,No,Non-Travel,144,Sales,10,4,Life Sciences,4,Male,...,0,0.260870,0.285714,0.857143,Jovem,7317.50,1.393279,0,0,0


## Vamos separar as features (x) e target (y)

X = todas as variáveis que o modelo vai usar para aprender (features originais + novas features)

y = A variável que o modelo vai tentar prever (Attrition → 0 ou 1)

- X (maiusculo) -> é uma matriz (muitas colunas e linhas)
- y (minusculo) -> é um vetor (uma única coluna)

In [ ]:
# Colunas que nao devem entrar como features
colunas_remover = ['Attrition', 'AttritionFlag']

# Separar X e y
y = df['AttritionFlag']
X = df.drop(columns=colunas_remover)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\ny distribuição:")
print(y.value_counts(normalize=True).round(4) * 100)

# Podemos observar que a distribuição permaneceu a mesma do dataset original

X shape: (1000000, 41)
y shape: (1000000,)

y distribuição:
AttritionFlag
0    83.99
1    16.01
Name: proportion, dtype: float64


In [7]:
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()

print(f"Categóricas: {cat_cols}")
print(f"Numéricas: {num_cols}")
print()
print(f'Total de variáveis categóricas: {len(cat_cols)}')
print(f'Total de variáveis numéricas: {len(num_cols)}')

Categóricas: ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime', 'AgeGroup']
Numéricas: ['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'SatisfactionScore', 'CriticalSatisfactionFlag', 'RatioCareerCompany', 'RoleStagnation', 'YearsWithoutPromotion', 'IncomePerLevel', 'IncomeMedianJob', 'OverTime_encoded', 'RiskOvertimeDistance', 'RiskWorklifeOvertime']

Total de variáveis categóricas: 8
Total de variáveis numéricas: 33


In [8]:
# Vamos remover o OverTime_encoded , pois foi utilizado apenas na etapa de calculo na etapa de feature engineering
# OverTime_encoded e OverTime são as mesmas informações, manter as duas gera redundância, pois vamos aplicar a técnica de OneHotEncoder em OverTime (yes, no).

X = X.drop(columns=['OverTime_encoded'])
num_cols.remove('OverTime_encoded')

print(f"X shape atualizado: {X.shape}")
print(f"Numéricas: {len(num_cols)}")
print(f"Categóricas: {len(cat_cols)}")

X shape atualizado: (1000000, 40)
Numéricas: 32
Categóricas: 8


### Anotações Pessoais

### Diferenças entre X_train, X_test, y_train, y_test**

**Treinamento**

É com os dois conjuntos X_train e y_train que o modelo vai passar pelo processo de aprendizagem. Nós passamos para ele as features e o resultado real de attrtition, como se fosse em
um aluno estudando para uma prova com um gabarito em mãos. Nessa etapa o modelo vai aprender com as features e os resultados reais dos funcionários.

**Teste**

Após a etapa de treinamento acima, vamos precisar descobrir se o modelo aprendeu a identificar os funcionários que podem entrar em attrition. Para esse cenário, vamos trabalhar com o conjunto X_test e sem apresentar 
a resposta real para o modelo. Ele simplesmente vai ler as features e tentar realizar a previsão de attrition.

Nessa etapa queremos ver o que o modelo aprendeu. E para validar essas respostas, usaremos o conjunto y_test, que seria o gabarito real. Ainda no exemplo do aluno, a etapa de teste
é como se fosse a prova, e o aluno vai responder as questões sem o gabarito, e no final será feito a correção para ver qual foi a nota do aluno com base no gabarito com todas as respostas corretas.

### Por qual motivo manter as mesmas distribuições no split?

É necessário manter a mesma distribuição do dataset real na etapa de split treino e teste para que o modelo aprenda com base no dataset real e não de forma aleatória. 

Dataset original:  84% No / 16% Yes (1.000.000 registros)

Split aleatório SEM stratify pode gerar:
- Treino: 90% No / 10% Yes  = poucos exemplos de Yes para aprender
- Teste:  70% No / 30% Yes  = muitos exemplos de Yes para avaliar

Com esse desbalanceamento no split aleatório podemos enviesar a resposta do modelo, já que ele aprendeu com peso maior em uma categoria.

In [ ]:
# Split treino/teste (80/20) com estratificação para manter as proporções corretas entre yes e no

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"\ny_train distribuição:")
print(y_train.value_counts(normalize=True).round(4) * 100)
print(f"\ny_test distribuição:")
print(y_test.value_counts(normalize=True).round(4) * 100)

# Podemos analisar que mantivemos a distibuição do dataset real.

X_train: (800000, 40)
X_test:  (200000, 40)

y_train distribuição:
AttritionFlag
0    83.99
1    16.01
Name: proportion, dtype: float64

y_test distribuição:
AttritionFlag
0    83.99
1    16.01
Name: proportion, dtype: float64


In [ ]:
# TODO: Detalhar sobre as técnicas de transformação nas variáveis e aplicações de encondings